# Minimal Pair 생성 파일럿 

Cell A에서 GPT API로 B, C, D 변형을 생성.

In [ ]:
import os
import json
import time
import pandas as pd
import ast
import re
from openai import OpenAI
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
 
# ============================================================
# 설정
# ============================================================
ANCHOR_PATH = "./data/processed/cell_a_anchors_v2_framed.csv" 
OUTPUT_PATH = "./data/processed/minimal_pairs_pilot_final_framed.csv"

PILOT_N = 30
MAX_RETRIES = 2

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MODEL = 'gpt-4o'

client = OpenAI(api_key=OPENAI_API_KEY)
vader = SentimentIntensityAnalyzer()

print(f"모델: {MODEL}")
print(f"파일럿 건수: {PILOT_N}")

모델: gpt-4o
파일럿 건수: 30


## 1. 로드 및 파일럿 샘플 선택

In [ ]:
df = pd.read_csv(ANCHOR_PATH)

# 리스트 컬럼 파싱
list_cols = ["tokens", "targets", "cue_tokens", "slur_tokens",
             "non_slur_cue_tokens", "target_tokens", "framing"]
for col in list_cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# ============================================================
# 필터: 만장일치 + Other 단독 타겟 제외
# ============================================================
print(f"전체 앵커: {len(df)}건")

# 만장일치만
df = df[df["agreement"] == "unanimous"].copy()
print(f"만장일치 필터 후: {len(df)}건")

# Other 단독 타겟 제외 (Other만 있고 다른 구체적 타겟이 없는 경우)
def has_specific_target(targets):
    specific = [t for t in targets if t != "Other"]
    return len(specific) > 0

df = df[df["targets"].apply(has_specific_target)].copy()
print(f"Other 단독 제외 후: {len(df)}건")


# 리스트 내 첫 번째 값을 대표 프레이밍(primary_framing)으로 추출
df['primary_framing'] = df['framing'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'NONE')

print(f"\n프레이밍 분포:")
print(df["primary_framing"].value_counts())

# 프레이밍 종류 확인 및 카테고리별 추출 할당량 계산
unique_framings = df['primary_framing'].unique()
n_per_framing = (PILOT_N // len(unique_framings)) + 1 

samples = []
for framing in unique_framings:
    framing_df = df[df['primary_framing'] == framing]
    # 해당 프레이밍의 전체 데이터가 할당량보다 적으면 있는 만큼만 전부 추출
    n_sample = min(n_per_framing, len(framing_df))
    samples.append(framing_df.sample(n_sample, random_state=42))

# 각 프레이밍별 샘플을 하나로 합치고 순서를 랜덤하게 섞음
pilot_df = pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)

# 설정한 PILOT_N(30건)에 정확히 맞게 상위 N개만 남김
pilot_df = pilot_df.head(PILOT_N)

print(f"\n파일럿 샘플: {len(pilot_df)}건")
print(pilot_df['primary_framing'].value_counts())

전체 앵커: 978건
만장일치 필터 후: 978건
Other 단독 제외 후: 925건

프레이밍 분포:
primary_framing
INTELLECTUAL_INFERIORITY    188
THREAT_VIOLENCE             164
GENERALIZATION              120
MORAL_DISGUST               112
DEHUMANIZATION              110
SEXUAL_GENDERED              81
CONSPIRACY                   55
CRIMINAL_DANGER              49
EXCLUSION                    35
ECONOMIC_BURDEN               8
RELIGIOUS                     3
Name: count, dtype: int64

파일럿 샘플: 30건
primary_framing
RELIGIOUS                   3
MORAL_DISGUST               3
ECONOMIC_BURDEN             3
INTELLECTUAL_INFERIORITY    3
GENERALIZATION              3
SEXUAL_GENDERED             3
THREAT_VIOLENCE             3
CRIMINAL_DANGER             3
DEHUMANIZATION              2
EXCLUSION                   2
CONSPIRACY                  2
Name: count, dtype: int64


## 2. 프롬프트 및 API 호출

In [30]:
SYSTEM_PROMPT = """You generate minimal pair sentences for NLP research on hate speech vs negative sentiment.

Given a hateful sentence (Cell A: strong polarity cue + demographic target) and its metadata (including the specific 'targets'), generate three JSON variants:

CELL B (Target Removed / Strong Cue):
- Keep profanity/negativity intensity equivalent to the original.
- Remove ALL group references. Redirect to a non-identity target.
- CRITICAL: Do NOT mechanically replace human targets with inanimate objects (e.g., "machines", "objects"). Use contextually appropriate, natural non-identity human groups (e.g., "corrupt politicians", "criminals", "anonymous trolls", "hypocrites").
- Identify and remove ANY dog-whistle terms or implicit targeting (e.g., "monkeys", "rats"). Replace them with generic insults (e.g., "idiots", "scum") that do not point to any protected demographic.
- Ensure the output reads like a natural, aggressive social media post.

CELL C (Slur Removed / Target Present / Implicit Hate):
- MANDATORY: You MUST explicitly use the EXACT demographic group(s) specified in the input 'targets' metadata.
- Convert the given target into specific, neutral terminology (e.g., 'African' -> "Black people", 'Islam' -> "Muslim people", 'Jewish' -> "Jewish people").
- FORBIDDEN: Do NOT invent targets. Do NOT use vague terms like "certain communities" or "some people".
- Remove ALL profanity, slurs, and explicit negative words.
- DYNAMIC FRAMING: You MUST use the logic of the original text's [framing] to structure your new sentence. Instead of just stating an opinion, rephrase the text so that it reflects the core concern of the [framing].
- CRITICAL: AVOID generic and repetitive starting templates like "I think...", "Why is it that...", or "Everyone knows...". Ensure high syntactic diversity.
- Convey prejudice through implication, stereotype, or dog-whistle language based ONLY on the original target, sounding like a natural social media user, NOT an academic observer.

CRITICAL TONE INSTRUCTION FOR CELL C:
- Write as if you are a regular person casually expressing a prejudiced opinion, NOT an academic observer.
- TONE (TWITTER STYLE): Convey prejudice through implication, stereotype, or dog-whistle language based ONLY on the original target. Make it sound EXACTLY like a real, casual Twitter (X) post. Keep it short, punchy, and conversational. Feel free to use typical internet phrasing, lower-case typing, or omit formal punctuation. NEVER sound like an academic observer.
- Do NOT use formal/analytical phrases like "It's essential to understand...", "There seems to be a trend...", "Some critics suggest..."

Examples of GOOD Cell C tone (real implicit hate from social media):
- "why are we always told we should be more tolerant of islam? why don't we ever hear that islam needs to be more tolerant of us"
- "black people have never formed a civilized society on their own without the help of white people"
- "jews should pay reparations to whites for white genocide"
- "so we should just let them come here do as they please and we foot the bill! not!"
- "maybe you should build your thriving black communities far away where white people can't get to them"
- "honestly women just aren't cut out for leadership roles, it's biology"
- "lol this isn't even your country. search your heritage, go back there."

Examples of BAD Cell C tone (too academic — AVOID THIS):
- "There seems to be a trend where Muslim people often face challenges integrating..."
- "It's essential to understand that African communities have different cultural values..."
- "Some believe that Jewish people have disproportionate influence in financial sectors..."
- "Research suggests that women may face unique challenges in professional settings..."

CELL D (Control / Neutralized):
- Create a completely neutral control sentence by removing BOTH the identity targets and all offensive language/slurs.
- CRITICAL: DO NOT change the core topic to something completely unrelated and trivial (e.g., do not change a political or vaccine discussion into a sentence about video games or the weather).
- MAINTAIN the original domain and core subject matter (e.g., public health, social policy, economic systems).
- Transform the original aggressive and biased text into a constructive, objective criticism or a neutral discussion within that exact same domain.
- Keep the sentence structure and length roughly similar to the original text.

Respond ONLY with valid JSON. Do NOT wrap in markdown code fences.
{
  "cell_b": "...",
  "cell_c": "...",
  "cell_d": "..."
}"""


def build_user_prompt(row):
    """앵커 1건에 대한 user 프롬프트 생성"""
    text = row.get("text_clean", row["text"])
    
    prompt = f"""ORIGINAL SENTENCE: {text}

    METADATA:
    - Polarity cue tokens (preserve in B): {row['non_slur_cue_tokens']}
    - Slur tokens (remove/replace in B, remove in C/D): {row['slur_tokens']}
    - Target tokens (remove in B/D, convert to neutral group names in C): {row['target_tokens']}
    - Target groups (MUST appear by name in Cell C): {row['targets']}
    - Framing type: {row['framing']}
    - Speech act: {row['speech_act']}
    - Slur is sole target marker: {row['slur_is_sole_target']}"""
    
    return prompt


def strip_markdown_fences(text):
    """마크다운 코드블록 제거"""
    clean = text.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1] if "\n" in clean else clean[3:]
    if clean.endswith("```"):
        clean = clean[:-3]
    return clean.strip()


def call_api(row, attempt=1):
    """Responses API 호출 + JSON 파싱"""
    user_prompt = build_user_prompt(row)

    try:
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=[{"role": "user", "content": user_prompt}],
            max_output_tokens=4096,
        )

        raw = response.output_text

        # 빈 응답 체크
        if not raw or len(raw.strip()) < 10:
            return {"success": False, "error": "Empty response (likely safety filter)", "raw": raw, "attempt": attempt}

        # 마크다운 코드블록 제거 후 JSON 파싱
        clean = strip_markdown_fences(raw)
        parsed = json.loads(clean)
        return {"success": True, "data": parsed, "raw": raw, "attempt": attempt}

    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSON parse error: {e}", "raw": raw, "attempt": attempt}
    except Exception as e:
        return {"success": False, "error": str(e), "raw": None, "attempt": attempt}


# 프롬프트 예시 출력
print("=== 프롬프트 예시 ===")
print(build_user_prompt(pilot_df.iloc[0]))

=== 프롬프트 예시 ===
ORIGINAL SENTENCE: <user> i long for the days where faggots who tried to erect some satanic statue wouldve been shot on the spot

    METADATA:
    - Polarity cue tokens (preserve in B): ['shot']
    - Slur tokens (remove/replace in B, remove in C/D): ['faggots']
    - Target tokens (remove in B/D, convert to neutral group names in C): ['faggots']
    - Target groups (MUST appear by name in Cell C): ['Other', 'Homosexual', 'Heterosexual', 'Minority']
    - Framing type: ['RELIGIOUS']
    - Speech act: OTHER
    - Slur is sole target marker: True


## 3. 자동 검증

In [31]:
# ============================================================
# Lexicon 정의
# ============================================================
PROFANITY_SET = {
    "fuck", "fucking", "fucked", "fucker", "fuckers", "fucks",
    "shit", "shitty", "shitting", "bullshit",
    "ass", "asshole", "assholes",
    "damn", "damned", "goddamn",
    "bitch", "bitches", "bitching",
    "bastard", "bastards",
    "crap", "crappy",
    "hell", "piss", "pissed",
}

NEG_EMOTION_SET = {
    "hate", "hated", "hating", "hatred",
    "disgusting", "disgust", "disgusted",
    "stupid", "stupidity",
    "ugly", "pathetic", "trash", "garbage",
    "horrible", "terrible", "awful", "vile",
    "nasty", "gross", "sick", "sickening",
    "loser", "losers", "idiot", "idiots", "moron", "morons",
    "dumb", "retarded", "retard",
    "worthless", "useless", "degenerate",
    "fool", "clown", "traitor",
}

VIOLENCE_SET = {
    "kill", "killed", "killing", "kills",
    "murder", "murdered", "murdering",
    "die", "dies", "dying", "dead", "death",
    "shoot", "shot", "shooting",
    "hang", "hanged", "hanging",
    "bomb", "bombing", "destroy", "exterminate",
    "stab", "stabbing", "slaughter",
    "rape", "raped", "raping", "abused", "abusing", "abuse",
}

MORAL_DISGUST_SET = {
    "filthy", "dirty", "scum", "vermin",
    "abomination", "degenerate", "depraved",
    "subhuman", "inhuman", "animal", "animals",
    "cockroach", "cockroaches", "parasite", "parasites",
    "rat", "rats", "monkey", "monkeys", "ape", "apes",
    "filth", "smelly", "coward",
}

POLARITY_CUE_LEXICON = PROFANITY_SET | NEG_EMOTION_SET | VIOLENCE_SET | MORAL_DISGUST_SET

# Cell B용 확장 cue 셋
BROAD_CUE_SET = POLARITY_CUE_LEXICON | {
    "failing", "traitorous", "incompetent", "pathetic", "ridiculous",
    "annoying", "miserable", "atrocious", "obnoxious", "absurd",
    "appalling", "despicable", "wretched", "lousy", "dreadful",
    "inferior", "mediocre", "pitiful", "ruin", "wreck", "smash",
    "moronic", "idiotic", "brainless", "mindless", "senseless",
    "toxic", "repulsive", "revolting", "horrendous", "abysmal",
    "screw", "screwed", "suck", "sucks", "sucked", "clueless",
    
    "grudge", "rigged", "incompetence", "deceitful", "deceit",
    "pointless", "foolish", "corrupt", "corruption",
    "manipulate", "manipulative", "reckless", "useless",
}

# 슬러셋
SLUR_BASE = {
    "nigger", "nigga", "coon", "darkie", "sambo", "jigaboo", "spook",
    "sheboon", "dindu", "negress",
    "kike", "yid", "heeb",
    "raghead", "towelhead", "sandnigger", "moslem",
    "spic", "wetback", "beaner",
    "chink", "gook", "zipperhead", "chinaman",
    "faggot", "fag", "dyke", "homo", "tranny",
    "whore", "slut", "hoe", "thot", "cunt",
    "honky", "honkey", "cracker",
    "retard", "wog",
}

SLUR_SET = set(SLUR_BASE)
for s in list(SLUR_BASE):
    SLUR_SET.update({s + "s", s + "es", s + "z"})

IDENTITY_TERMS = {
    "black", "blacks", "african", "africans",
    "jewish", "jews", "jew",
    "muslim", "muslims", "islam", "islamic",
    "arab", "arabs", "arabic",
    "hispanic", "latino", "latina", "mexican", "mexicans",
    "asian", "asians", "chinese", "korean", "japanese",
    "gay", "gays", "lesbian", "lesbians", "homosexual", "homosexuals",
    "transgender", "trans", "queer", "lgbtq", "lgbt",
    "woman", "women", "female", "females",
    "man", "men", "male", "males",
    "white", "whites", "caucasian", "caucasians",
    "immigrant", "immigrants", "refugee", "refugees",
    "christian", "christians", "hindu", "hindus",
    "disabled", "disability",
}

TARGET_LEXICON = IDENTITY_TERMS | SLUR_SET



# ============================================================
# 검증 함수
# ============================================================
def get_tokens_lower(text):
    return [t.lower() for t in text.split()]

def has_strong_cue(text):
    tokens = set(get_tokens_lower(text))
    return len(tokens & (POLARITY_CUE_LEXICON | SLUR_SET)) > 0

def has_broad_cue(text):
    """Cell B용: 넓은 부정어 셋으로 cue 존재 체크"""
    tokens = set(get_tokens_lower(text))
    return len(tokens & BROAD_CUE_SET) > 0

def has_target(text):
    tokens = set(get_tokens_lower(text))
    return len(tokens & TARGET_LEXICON) > 0

def has_identity_term(text):
    """Cell C용: identity term만 체크 (슬러 제외)"""
    tokens = set(get_tokens_lower(text))
    return len(tokens & IDENTITY_TERMS) > 0

def vader_clean(text):
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0\U000024C2-\U0001F251"
        "\U00010000-\U0010ffff\u2600-\u2B55]+", flags=re.UNICODE
    )
    clean = emoji_pattern.sub('', text).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    return vader.polarity_scores(clean)["compound"]


def validate_generation(anchor_row, generated):
    """생성된 B/C/D 자동 검증"""
    results = {}
    anchor_text = anchor_row.get("text_clean", anchor_row["text"])
    anchor_vader = vader_clean(anchor_text)
    anchor_tokens = len(anchor_text.split())

    # --- Cell B: strong cue + no target ---
    b_text = generated.get("cell_b", "")
    if b_text:
        b_vader = vader_clean(b_text)
        b_checks = {
            "has_cue": has_broad_cue(b_text),
            "no_target": not has_target(b_text),
            "vader_diff": round(abs(anchor_vader - b_vader), 3),
            "vader_ok": abs(anchor_vader - b_vader) < 0.5,  
            "length_ratio": round(len(b_text.split()) / max(anchor_tokens, 1), 2),
            "length_ok": 0.5 < len(b_text.split()) / max(anchor_tokens, 1) < 1.5,
        }
        b_checks["passed"] = b_checks["has_cue"] and b_checks["no_target"] and b_checks["length_ok"] and b_checks["vader_ok"]
    else:
        b_checks = {"passed": False, "error": "empty"}
    results["cell_b"] = b_checks

    # --- Cell C: weak cue + target present ---
    c_text = generated.get("cell_c", "")
    if c_text:
        c_checks = {
            "no_strong_cue": not has_strong_cue(c_text),
            "has_identity_term": has_identity_term(c_text),  
            "length_ratio": round(len(c_text.split()) / max(anchor_tokens, 1), 2),
            "length_ok": 0.5 < len(c_text.split()) / max(anchor_tokens, 1) < 2.5,
        }
        c_checks["passed"] = c_checks["no_strong_cue"] and c_checks["has_identity_term"] and c_checks["length_ok"]
    else:
        c_checks = {"passed": False, "error": "empty"}
    results["cell_c"] = c_checks

    # --- Cell D: weak cue + no target ---
    d_text = generated.get("cell_d", "")
    if d_text:
        d_checks = {
            "no_strong_cue": not has_strong_cue(d_text),
            "no_target": not has_target(d_text),
            "length_ratio": round(len(d_text.split()) / max(anchor_tokens, 1), 2),
            "length_ok": 0.5 < len(d_text.split()) / max(anchor_tokens, 1) < 1.5,
        }
        d_checks["passed"] = d_checks["no_strong_cue"] and d_checks["no_target"] and d_checks["length_ok"]
    else:
        d_checks = {"passed": False, "error": "empty"}
    results["cell_d"] = d_checks

    return results


# 검증 함수 테스트
print("검증 함수 로드 완료")
print(f"  'fucking idiot' broad_cue: {has_broad_cue('fucking idiot')}")
print(f"  'incompetent fool' broad_cue: {has_broad_cue('incompetent fool')}")
print(f"  'the weather is nice' broad_cue: {has_broad_cue('the weather is nice')}")
print(f"  'Black people tend to...' identity: {has_identity_term('Black people tend to...')}")
print(f"  'certain communities' identity: {has_identity_term('certain communities')}")

검증 함수 로드 완료
  'fucking idiot' broad_cue: True
  'incompetent fool' broad_cue: True
  'the weather is nice' broad_cue: False
  'Black people tend to...' identity: True
  'certain communities' identity: False


## 4. 생성 실행

In [ ]:
def run_generation_with_retry(row, max_retries=MAX_RETRIES):
    """1건 생성 + 자동 검증 + 재생성"""
    for attempt in range(1, max_retries + 2):
        result = call_api(row, attempt=attempt)

        if not result["success"]:
            print(f"    [attempt {attempt}] API 오류: {result['error']}")
            if attempt <= max_retries:
                time.sleep(2)
                continue
            else:
                return None, None, attempt

        generated = result["data"]
        validation = validate_generation(row, generated)
        all_passed = all(v.get("passed", False) for v in validation.values())

        if all_passed or attempt > max_retries:
            return generated, validation, attempt

        failed = [k for k, v in validation.items() if not v.get("passed", False)]
        print(f"    [attempt {attempt}] 검증 실패: {failed} — 재생성")
        time.sleep(1)

    return None, None, max_retries + 1


# === 파일럿 실행 ===
all_results = []

print(f"파일럿 생성 시작: {PILOT_N}건")
print("=" * 60)

for i, (idx, row) in enumerate(pilot_df.iterrows()):
    print(f"\n[{i+1}/{len(pilot_df)}] {row['post_id']} ({row['majority_label']}, {row['primary_framing']})")
    print(f"  A: {row.get('text_clean', row['text'])[:100]}...")

    generated, validation, attempts = run_generation_with_retry(row)

    if generated is None:
        print(f"  ✗ 생성 실패 ({attempts}회 시도)")
        all_results.append({"post_id": row["post_id"], "status": "failed", "attempts": attempts})
        continue

    # 결과 저장
    entry = {
        "post_id": row["post_id"],
        "status": "success",
        "attempts": attempts,
        "primary_framing": row["primary_framing"], 
        "majority_label": row["majority_label"],
        "text_a": row.get("text_clean", row["text"]),
        "targets": str(row["targets"]),
        "framing": str(row["framing"]),
        "speech_act": row["speech_act"],
    }

    for cell in ["cell_b", "cell_c", "cell_d"]:
        cell_text = generated.get(cell, "")
        cell_val = validation.get(cell, {})
        
        entry[f"{cell}_text"] = cell_text
        entry[f"{cell}_passed"] = cell_val.get("passed", False)
        entry[f"{cell}_checks"] = json.dumps(
            {k: v for k, v in cell_val.items() if k != "passed"},
            ensure_ascii=False, default=str
        )

    all_results.append(entry)

    b_mark = "✓" if validation["cell_b"].get("passed") else "✗"
    c_mark = "✓" if validation["cell_c"].get("passed") else "✗"
    d_mark = "✓" if validation["cell_d"].get("passed") else "✗"
    
    print(f"  B{b_mark}: {generated.get('cell_b', '')[:80]}...")
    print(f"  C{c_mark}: {generated.get('cell_c', '')[:80]}...")
    print(f"  D{d_mark}: {generated.get('cell_d', '')[:80]}...")
    print(f"  ({attempts}회 시도)")

    time.sleep(0.5)

print("\n" + "=" * 60)
print(f"생성 완료: {len(all_results)}건")


파일럿 생성 시작: 30건

[1/30] 1178360025993142275_twitter (hatespeech, RELIGIOUS)
  A: <user> i long for the days where faggots who tried to erect some satanic statue wouldve been shot on...
    [attempt 1] 검증 실패: ['cell_b'] — 재생성
    [attempt 2] 검증 실패: ['cell_b'] — 재생성
  B✗: i long for the days where idiots who tried to erect some satanic statue wouldve ...
  C✓: those homosexual people putting up statues like that, always pushing the boundar...
  D✓: some individuals who tried to erect a controversial statue could have faced stro...
  (3회 시도)

[2/30] 18431319_gab (hatespeech, MORAL_DISGUST)
  A: all these god dam filthy savages and nigger muslim dothaids these fuckin babyraping jews keep bringi...
  B✓: all these filthy idiots and criminals these babyraping scum keep bringing in is ...
  C✓: muslim people and black people keep bringing in issues and jewish people are at ...
  D✓: the increase in population movements is contributing to a flu epidemic. public h...
  (1회 시도)

[3/30] 4701852_ga

## 5. 결과 분석

In [ ]:
results_df = pd.DataFrame(all_results)
success = results_df[results_df["status"] == "success"]
failed = results_df[results_df["status"] == "failed"]

print(f"성공: {len(success)}건 / 실패: {len(failed)}건")

if len(success) > 0:
    # 셀별 통과율
    print(f"\n셀별 자동 검증 통과율:")
    for cell in ["cell_b", "cell_c", "cell_d"]:
        passed = success[f"{cell}_passed"].sum()
        print(f"  {cell}: {passed}/{len(success)} ({passed/len(success)*100:.1f}%)")

    all_pass = (success["cell_b_passed"] & success["cell_c_passed"] & success["cell_d_passed"]).sum()
    print(f"  전체 통과: {all_pass}/{len(success)} ({all_pass/len(success)*100:.1f}%)")

    # 프레이밍별
    print(f"\n프레이밍별 전체 통과율:")
    for framing in success["primary_framing"].unique():
        sub = success[success["primary_framing"] == framing]
        if len(sub) > 0:
            ap = (sub["cell_b_passed"] & sub["cell_c_passed"] & sub["cell_d_passed"]).sum()
            print(f"  {framing}: {ap}/{len(sub)} ({ap/len(sub)*100:.1f}%)")

    print(f"\n평균 시도 횟수: {success['attempts'].mean():.1f}")

    # VADER A vs B
    print(f"\nVADER 비교 (A vs B):")
    for _, row in success.head(10).iterrows():
        if row["cell_b_text"]:
            v_a = vader_clean(row["text_a"])
            v_b = vader_clean(row["cell_b_text"])
            diff = abs(v_a - v_b)
            flag = " ⚠" if diff > 0.25 else ""
            print(f"  A={v_a:.3f} B={v_b:.3f} Δ={diff:.3f}{flag}")
            print(f"    A: {row['text_a'][:70]}...")
            print(f"    B: {row['cell_b_text'][:70]}...")

성공: 30건 / 실패: 0건

셀별 자동 검증 통과율:
  cell_b: 29/30 (96.7%)
  cell_c: 29/30 (96.7%)
  cell_d: 30/30 (100.0%)
  전체 통과: 28/30 (93.3%)

프레이밍별 전체 통과율:
  RELIGIOUS: 2/3 (66.7%)
  MORAL_DISGUST: 3/3 (100.0%)
  ECONOMIC_BURDEN: 2/3 (66.7%)
  INTELLECTUAL_INFERIORITY: 3/3 (100.0%)
  DEHUMANIZATION: 2/2 (100.0%)
  GENERALIZATION: 3/3 (100.0%)
  SEXUAL_GENDERED: 3/3 (100.0%)
  EXCLUSION: 2/2 (100.0%)
  THREAT_VIOLENCE: 3/3 (100.0%)
  CRIMINAL_DANGER: 3/3 (100.0%)
  CONSPIRACY: 2/2 (100.0%)

평균 시도 횟수: 1.4

VADER 비교 (A vs B):
  A=-0.637 B=0.000 Δ=0.637 ⚠
    A: <user> i long for the days where faggots who tried to erect some satan...
    B: i long for the days where idiots who tried to erect some satanic statu...
  A=-0.952 B=-0.954 Δ=0.002
    A: all these god dam filthy savages and nigger muslim dothaids these fuck...
    B: all these filthy idiots and criminals these babyraping scum keep bring...
  A=0.852 B=0.844 Δ=0.008
    A: fanbloodytastic by time these leeches are held to account especially t

## 6. 수동 검토

In [34]:
if len(success) > 0:
    print("=" * 70)
    print("수동 검토용 — 생성된 minimal pairs")
    print("=" * 70)

    for i, (_, row) in enumerate(success.iterrows()):
        print(f"\n{'─' * 70}")
        print(f"[{i+1}] {row['post_id']} | {row['majority_label']} | {row['primary_framing']}")
        print(f"  targets: {row['targets']}")
        print(f"  framing: {row['framing']} | speech_act: {row['speech_act']}")
        print()

        b_mark = "✓" if row["cell_b_passed"] else "✗"
        c_mark = "✓" if row["cell_c_passed"] else "✗"
        d_mark = "✓" if row["cell_d_passed"] else "✗"

        print(f"  A (원문):     {row['text_a']}")
        print(f"  B (cue+){b_mark}:  {row['cell_b_text']}")
        print(f"  C (tgt+){c_mark}:  {row['cell_c_text']}")
        print(f"  D (중립){d_mark}:  {row['cell_d_text']}")
        print()

수동 검토용 — 생성된 minimal pairs

──────────────────────────────────────────────────────────────────────
[1] 1178360025993142275_twitter | hatespeech | RELIGIOUS
  targets: ['Other', 'Homosexual', 'Heterosexual', 'Minority']
  framing: ['RELIGIOUS'] | speech_act: OTHER

  A (원문):     <user> i long for the days where faggots who tried to erect some satanic statue wouldve been shot on the spot
  B (cue+)✗:  i long for the days where idiots who tried to erect some satanic statue wouldve been shot on the spot
  C (tgt+)✓:  those homosexual people putting up statues like that, always pushing the boundaries of values in our society
  D (중립)✓:  some individuals who tried to erect a controversial statue could have faced strong opposition in the past


──────────────────────────────────────────────────────────────────────
[2] 18431319_gab | hatespeech | MORAL_DISGUST
  targets: ['African', 'Jewish', 'Islam']
  framing: ['MORAL_DISGUST', 'GENERALIZATION'] | speech_act: OTHER

  A (원문):     all these g

## 7. 검증 실패 진단

In [35]:
if len(success) > 0:
    for cell in ["cell_b", "cell_c", "cell_d"]:
        failed_cell = success[~success[f"{cell}_passed"]]
        if len(failed_cell) > 0:
            print(f"\n{'=' * 60}")
            print(f"{cell} 검증 실패: {len(failed_cell)}건")
            print(f"{'=' * 60}")
        else:
            print(f"\n{cell}: 실패 0건 ✓")


cell_b 검증 실패: 1건

cell_c 검증 실패: 1건

cell_d: 실패 0건 ✓


## 8. 저장

In [36]:
results_df.to_csv(OUTPUT_PATH, index=False)
print(f"저장: {OUTPUT_PATH}")
print(f"총 {len(results_df)}건 ({len(success)} 성공, {len(failed)} 실패)")

저장: ./data/processed/minimal_pairs_pilot_final_framed.csv
총 30건 (30 성공, 0 실패)
